In [ ]:
import pandas as pd 
import os 


In [ ]:
class OneInstance:
    def __init__(self, row):
        self.epsilon_value = float(row['epsilon_value'])
        self.result = row.get('result', None)
        self.time = row.get('time', None)
        self.verifier = row.get('verifier', None)
        self.network_name = row.get('network_name', None)
        self.image_name = row.get('image_name', None)
        self.mps_path = row.get('mps_path', None)
    def __repr__(self):
        return (f"OneInstance(net={self.network_name}, img={self.image_name}, "
                f"eps={self.epsilon_value}, res={self.result})")

class TwoInstance:
    def __init__(self, first_instance: OneInstance, second_instance: OneInstance):
        self.first = first_instance
        self.second = second_instance
        # placeholders for outputs after running the warmstart experiment
        self.second_result = None
        self.second_time = None
    def __repr__(self):
        return f"TwoInstance({self.first} -> {self.second})"


In [ ]:
base_path = '/home/annelot/WARMSTART_PROJECT/baseline_symphony_14-01-2026+12_30/tmp'

df_list = []


for network_name in os.listdir(base_path):
    network_path = os.path.join(base_path, network_name)
    

    if os.path.isdir(network_path):
        for image_name in os.listdir(network_path):
            split_path = os.path.join(network_path, image_name)
   
            csv_path = os.path.join(split_path, "epsilons_df.csv")
            
            if os.path.exists(csv_path):
                # Read the distribution CSV
                df = pd.read_csv(csv_path)
                df['network_name'] = network_name
                df['image_name'] = image_name

                df['mps_path']  = split_path+ "/"+ network_name+ "_"+ image_name.removeprefix("image_")+ "_"+ df["epsilon_value"].astype(str).str.replace(".", "_", regex=False)+ ".mps"
                df_list.append(df)
                
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_csv("baseline_results_per_epsilon.csv", index=False)


In [ ]:

df = combined_df

In [ ]:
instances = {}
required_cols = ['epsilon_value','result','time','verifier','network_name','image_name','mps_path']

for _, row in df.iterrows():
    one = OneInstance(row)
    net = one.network_name
    img = one.image_name
    eps = one.epsilon_value
    instances.setdefault(net, {}).setdefault(img, {})[eps] = one


def sorted_eps_list(mapping_eps_to_oneinstance):
    # returns sorted list of eps values (ascending)
    return sorted(mapping_eps_to_oneinstance.keys())

def safe_get_instance(mapping, net, img, eps):
    try:
        return mapping[net][img][eps]
    except KeyError:
        return None


    

In [ ]:
new_experiments = []

df = df[df['result'] != 'ERR']

for net, images_dict in instances.items():
    for img, eps_map in images_dict.items():
        eps_sorted = sorted_eps_list(eps_map)
        if not eps_sorted:
            continue

        # slice the df for this (net,img) to extract results and ensure sort
        sliced = df[(df['network_name'] == net) & (df['image_name'] == img)].copy()
        sliced['epsilon_value'] = sliced['epsilon_value'].astype(float)
        sliced = sliced.sort_values('epsilon_value')

        # lists of eps by result label
        unsats = sliced[sliced['result'] == 'UNSAT']['epsilon_value'].tolist()
        sats = sliced[sliced['result'] == 'SAT']['epsilon_value'].tolist()
        timeouts_or_missing = sliced[sliced['result'].isnull() | (sliced['result']=='TIMEOUT')]['epsilon_value'].tolist()

        for net in instances:
            for img in instances[net]:

                sliced = df[(df['network_name'] == net) &
                            (df['image_name'] == img)].sort_values('epsilon_value')

                unsats = sliced[sliced['result'] == 'UNSAT']['epsilon_value'].to_list()
                sats   = sliced[sliced['result'] == 'SAT']['epsilon_value'].to_list()

                
                if len(unsats)>1:
                    # use last two UNSAT eps
                    eps_a, eps_b = unsats[-2], unsats[-1]
                    inst_a = instances[net][img][eps_a]
                    inst_b = instances[net][img][eps_b]

                    if inst_a is not inst_b:
                        new_experiments.append(TwoInstance(inst_a, inst_b))

                if len(sats) > 1:
                    inst_a = instances[net][img][sats[1]]
                    inst_b = instances[net][img][sats[0]]
                    if inst_a is not inst_b:
                        new_experiments.append(TwoInstance(inst_a, inst_b))


        for missing_eps in timeouts_or_missing:
            candidates = [e for e in eps_sorted if e not in timeouts_or_missing]
            if not candidates:
                continue
           
            nearest = min(candidates, key=lambda x: abs(x - missing_eps))
            solved_inst = safe_get_instance(instances, net, img, nearest)
            missing_inst = safe_get_instance(instances, net, img, float(missing_eps))
            if solved_inst and missing_inst:
                new_experiments.append(TwoInstance(solved_inst, missing_inst))

    
        for eps in eps_sorted:
            other_imgs = [other_img for other_img in instances[net] if other_img != img and eps in instances[net][other_img]]
            if other_imgs:
                other_inst = instances[net][other_imgs[0]][eps]
                this_inst = eps_map[eps]
                new_experiments.append(TwoInstance(this_inst, other_inst))


image_index = {}
for net, images_dict in instances.items():
    for img, eps_map in images_dict.items():
        for eps, one in eps_map.items():
            image_index.setdefault(img, {}).setdefault(eps, []).append((net, one))

for img, eps_map in image_index.items():
    for eps, net_list in eps_map.items():
        if len(net_list) >= 2:
            # pair first two networks for an experiment
            inst_a = net_list[0][1]
            inst_b = net_list[1][1]
            new_experiments.append(TwoInstance(inst_a, inst_b))

seen = set()
unique_experiments = []
for t in new_experiments:
    sig = (t.first.network_name, t.first.image_name, t.first.epsilon_value,
           t.second.network_name, t.second.image_name, t.second.epsilon_value)
    if sig not in seen:
        seen.add(sig)
        unique_experiments.append(t)

print(f"Generated {len(unique_experiments)} unique warmstart experiment pairs.")

unique_experiments[:20]
        
        

